<a href="https://colab.research.google.com/github/dayvidwesley/Projetos-PEP/blob/main/porsche_go_analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# POP — Análise Comercial e de Estoque (Porsche GO)
**Disciplina:** Práticas em Engenharia de Produção — FCT/UFG  
**Autoras:** Ana Laura Clemente Tavares e Amanda Rosa Tavares

Este notebook executa o Procedimento Operacional Padrão (POP) para extração e análise de indicadores comerciais e de estoque a partir de dados exportados do Salesforce CRM, utilizando um banco de dados SQLite local.

**Etapas do POP:** Obtenção → Validação → Conexão → Ingestão (ETL) → Extração via SQL → Visualização

**Nota sobre os dados:** o arquivo `budget_ficticio.csv` usado neste notebook possui valores ilustrativos, pois os dados financeiros reais da unidade são confidenciais (restritos ao perfil de gestor no Salesforce, conforme LGPD). O arquivo de estoque é real.

## 1. Instalação das bibliotecas

In [ ]:
!pip install pandas matplotlib -q



```
# Isto está formatado como código
```

## 2. Etapa 1 — Obtenção dos Dados

Para rodar no Colab, suba os dois arquivos no menu lateral esquerdo:

- `estoque_04052026.csv` — exportado diretamente do Salesforce pela própria autora.
- `budget_ficticio.csv` — estrutura idêntica ao budget real, com valores ilustrativos (o budget real é restrito ao perfil de gestor e não é divulgado por confidencialidade).

Se este notebook foi aberto a partir do repositório GitHub da turma, ambos os arquivos já estão na mesma pasta e este passo pode ser pulado.

## 3. Etapa 2 — Validação dos Arquivos Recebidos

In [ ]:
import os
import pandas as pd

arquivos_esperados = ["estoque_04052026.csv", "budget_ficticio.csv"]

for arq in arquivos_esperados:
    existe = os.path.exists(arq)
    status = "OK" if existe else "FALTANDO"
    print(f"[{status}] {arq}")

# Validação de encoding e colunas esperadas no estoque
df_check = pd.read_csv("estoque_04052026.csv", encoding="utf-8-sig", nrows=5)
colunas_esperadas = [
    "SITUAÇÃO", "MARCA", "MODELO", "ANO", "CHASSI",
    "PERC MARGEM %", "DIAS PÁTIO"
]

faltando = [c for c in colunas_esperadas if c not in df_check.columns]
if faltando:
    print(f"⚠ Colunas faltando no estoque: {faltando}")
else:
    print("✔ Todas as colunas esperadas estão presentes no estoque.")

print(f"✔ Encoding lido corretamente (utf-8-sig). {len(df_check)} linhas de amostra validadas.")

## 4. Etapa 3 — Conexão ao Banco de Dados

In [ ]:
import sqlite3
import matplotlib.pyplot as plt

# String de conexão ao banco SQLite local
DB_PATH = "porsche_go.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print(f"Conexão estabelecida: {DB_PATH}")
print(f"Versão SQLite: {sqlite3.sqlite_version}")

## 5. Etapa 4 — Ingestão dos Dados (ETL)

In [ ]:
# Tabela ESTOQUE
cursor.execute("DROP TABLE IF EXISTS estoque")
cursor.execute("""
    CREATE TABLE estoque (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        situacao TEXT, marca TEXT, modelo TEXT, ano TEXT, km REAL,
        chassi TEXT UNIQUE, cor_externa TEXT, preco_venda REAL,
        custo_total REAL, margem REAL, perc_margem REAL, dias_patio INTEGER
    )
""")

df_est = pd.read_csv("estoque_04052026.csv", encoding="utf-8-sig")

df_est_db = df_est[[
    "SITUAÇÃO", "MARCA", "MODELO", "ANO", "KM", "CHASSI",
    "COR EXTERNA", "PREÇO VENDA", "CUSTO TOTAL", "MARGEM",
    "PERC MARGEM %", "DIAS PÁTIO"
]].copy()

df_est_db.columns = [
    "situacao", "marca", "modelo", "ano", "km", "chassi",
    "cor_externa", "preco_venda", "custo_total", "margem",
    "perc_margem", "dias_patio"
]

df_est_db["dias_patio"] = pd.to_numeric(df_est_db["dias_patio"], errors="coerce")
df_est_db["perc_margem"] = pd.to_numeric(
    df_est_db["perc_margem"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)
df_est_db["km"] = pd.to_numeric(df_est_db["km"], errors="coerce")

df_est_db.to_sql("estoque", conn, if_exists="append", index=False)
print(f"Tabela ESTOQUE criada: {len(df_est_db)} registros carregados")

In [ ]:
# Tabela BUDGET — lida a partir do CSV (fictício, estrutura idêntica ao real)
cursor.execute("DROP TABLE IF EXISTS budget")
cursor.execute("""
    CREATE TABLE budget (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        indicador TEXT,
        orc_jan REAL, ef_jan REAL,
        orc_fev REAL, ef_fev REAL,
        orc_mar REAL, ef_mar REAL
    )
""")

df_bud = pd.read_csv("budget_ficticio.csv", encoding="utf-8-sig")
df_bud.to_sql("budget", conn, if_exists="append", index=False)
print(f"Tabela BUDGET criada: {len(df_bud)} indicadores carregados")

## 6. Etapa 5 — Extração via SQL
### Indicador 1 — Variação Orçado vs. Realizado (Março)

In [ ]:
query_var = """
    SELECT
        indicador,
        orc_mar AS orcado,
        ef_mar AS realizado,
        ROUND((ef_mar - orc_mar) / ABS(orc_mar) * 100, 1) AS variacao_pct
    FROM budget
    WHERE indicador IN (
        'Volume Novos',
        'Faturamento Total Veículos',
        'Margem Total Veículos',
        'Resultado Comercial'
    )
    ORDER BY variacao_pct DESC
"""

df_var = pd.read_sql(query_var, conn)
df_var

### Indicador 2 — Veículos Críticos (Pátio > 90 dias)

In [ ]:
query_crit = """
    SELECT
        situacao, modelo, ano, dias_patio,
        ROUND(perc_margem, 2) AS margem_pct
    FROM estoque
    WHERE dias_patio > 90
    ORDER BY dias_patio DESC
"""

df_crit = pd.read_sql(query_crit, conn)
print(f"Total de veículos críticos: {len(df_crit)}")
df_crit

### Indicador 3 — Margem Média por Situação de Estoque

In [ ]:
query_sit = """
    SELECT
        situacao,
        COUNT(*) AS total_veiculos,
        ROUND(AVG(perc_margem), 2) AS margem_media_pct,
        ROUND(AVG(dias_patio), 1) AS dias_medio_patio
    FROM estoque
    GROUP BY situacao
    ORDER BY margem_media_pct
"""

df_sit = pd.read_sql(query_sit, conn)
df_sit

### Indicador 4 — Giro de Estoque

In [ ]:
vol_mar = df_var.loc[df_var["indicador"] == "Volume Novos", "realizado"].values[0]
total_est = len(df_est_db)

print(f"Vendas março: {int(vol_mar)} unidades")
print(f"Estoque total: {total_est} unidades")
print(f"Giro (vendas/estoque): {vol_mar/total_est:.2f}x")

## 7. Etapa 6 — Visualização dos Resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
fig.suptitle("Porsche GO — Análise Q1 2026", fontsize=13, fontweight="bold", y=1.02)

# ── Gráfico 1: Orçado vs. Realizado (eixo duplo: Volume em unidades x Valores em R$) ──
ax1 = axes[0]
df_vol = df_var[df_var["indicador"] == "Volume Novos"]
df_val = df_var[df_var["indicador"] != "Volume Novos"]

ax1b = ax1.twinx()

# Volume Novos no eixo esquerdo (unidades) — escala muito menor que os valores em R$
x_vol = [0]
ax1.bar([x_vol[0]-0.2], df_vol["orcado"], width=0.38, color="#1a237e", alpha=0.85, label="Orçado")
ax1.bar([x_vol[0]+0.2], df_vol["realizado"], width=0.38, color="#e65100", alpha=0.85, label="Realizado")
ax1.set_ylabel("Volume Novos (unidades)", fontsize=9, color="#333")

# Indicadores financeiros no eixo direito (R$)
x_val = [1, 2, 3]
ax1b.bar([i-0.2 for i in x_val], df_val["orcado"], width=0.38, color="#1a237e", alpha=0.85)
ax1b.bar([i+0.2 for i in x_val], df_val["realizado"], width=0.38, color="#e65100", alpha=0.85)
ax1b.set_ylabel("Valor (R$)", fontsize=9, color="#333")

all_x = x_vol + x_val
all_labels = ["Volume\nNovos", "Resultado\nComercial", "Margem Total\nVeículos", "Faturamento Total\nVeículos"]
ax1.set_xticks(all_x)
ax1.set_xticklabels(all_labels, fontsize=8.5)
ax1.set_xlim(-0.6, 3.6)
ax1.set_title("Orçado vs. Realizado — Março", fontsize=11)
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(axis="y", alpha=0.3)

# Rótulos de variação percentual acima de cada par de barras
for i, row in df_var.reset_index(drop=True).iterrows():
    if row["indicador"] == "Volume Novos":
        ax_use, x_pos = ax1, x_vol[0]
    else:
        ax_use = ax1b
        x_pos = x_val[list(df_val["indicador"]).index(row["indicador"])]
    color = "#2e7d32" if row["variacao_pct"] >= 0 else "#c62828"
    ymax = max(row["orcado"], row["realizado"])
    ax_use.text(x_pos, ymax * 1.04, f"{row['variacao_pct']:+.0f}%", ha="center", fontsize=9, color=color, fontweight="bold")

# ── Gráfico 2: Dias em Pátio por Veículo ────────────────────────────
ax2 = axes[1]
df_pat = pd.read_sql("""
    SELECT modelo || ' ' || ano AS veiculo, dias_patio
    FROM estoque ORDER BY dias_patio DESC
""", conn)
cores = df_pat["dias_patio"].apply(lambda d: "#c62828" if d > 120 else "#e65100" if d > 60 else "#1565c0")
ax2.barh(df_pat["veiculo"], df_pat["dias_patio"], color=cores, alpha=0.85)
ax2.set_xlabel("Dias em Pátio")
ax2.set_title("Estoque — Dias em Pátio por Veículo", fontsize=11)
ax2.axvline(x=90, color="#c62828", linestyle="--", linewidth=1, label="Limite crítico (90 dias)")
ax2.legend(fontsize=9)
ax2.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("resultado_porsche_go.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Encerrar Conexão

In [ ]:
conn.close()
print("Conexão encerrada.")

## 9. Evolução do Procedimento para Versões Futuras

O procedimento atual utiliza arquivos CSV exportados manualmente do Salesforce. Como evolução, o POP pode evoluir para acesso programático direto via API do Salesforce, utilizando a biblioteca `simple_salesforce` com queries em **SOQL** (Salesforce Object Query Language):

```python
from simple_salesforce import Salesforce

sf = Salesforce(
    username='usuario@empresa.com',
    password='senha',
    security_token='token'
)

resultado = sf.query("""
    SELECT Owner.Name, COUNT(Id) total, SUM(Amount) receita
    FROM Opportunity
    WHERE StageName = 'Closed Won' AND CloseDate = THIS_MONTH
    GROUP BY Owner.Name
""")

df = pd.DataFrame(resultado['records'])
```

Essa evolução elimina a dependência de exportações manuais, mas **não altera a lógica de análise** — apenas a camada de ingestão dos dados. Esse desacoplamento é o princípio da Clean Architecture (MARTIN, 2017) que fundamenta este POP: o banco de dados é um detalhe de implementação.